In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from tqdm.auto import tqdm

In [3]:
project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

clean_metadata_path = (
    processed_dir / "agrisense_metadata_clean.csv"
)

df_clean = pd.read_csv(clean_metadata_path)

print("Dataset shape:", df_clean.shape)
print("Disease classes:", df_clean["Disease"].nunique())

Dataset shape: (7399, 11)
Disease classes: 115


In [4]:
# Recreating disease mapping

disease_classes = sorted(df_clean["Disease"].unique())

disease_to_id = {
    disease: idx
    for idx, disease in enumerate(disease_classes)
}

id_to_disease = {
    idx: disease
    for disease, idx in disease_to_id.items()
}

df_clean["disease_id"] = df_clean["Disease"].map(disease_to_id)

print("Number of classes:", len(disease_classes))
print("Missing disease IDs:", df_clean["disease_id"].isna().sum())

Number of classes: 115
Missing disease IDs: 0


In [5]:
# Recreating actual imagr paths

image_root = (
    project_root
    / "data"
    / "raw"
    /"PlantSeg"
    /"images"
)

def find_actual_image_path(row):
    expected_path = image_root / row["Name"]

    if expected_path.exists():
        return expected_path

    matches = list(image_root.rglob(row["Name"]))

    if len(matches) == 1:
        return matches[0]

    return None

df_clean["actual_image_path"] = df_clean.apply(
    find_actual_image_path,
    axis=1
)

print(
    "Missing image paths:",
    df_clean["actual_image_path"].isna().sum()
)

Missing image paths: 0


In [6]:
#Recreating transforms and dataset

class ResizeWithPadding:
    def __init__(self, size=(224, 224)):
        self.size = size

    def __call__(self, image):
        target_width, target_height = self.size

        scale = min(
            target_width / image.width,
            target_height / image.height
        )

        new_width = int(image.width * scale)
        new_height = int(image.height * scale)

        image = image.resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS
        )

        canvas = Image.new(
            "RGB",
            self.size,
            (0, 0, 0)
        )

        left = (target_width - new_width) // 2
        top = (target_height - new_height) // 2

        canvas.paste(image, (left, top))

        return canvas

In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    ResizeWithPadding((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_test_transform = transforms.Compose([
    ResizeWithPadding((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

In [8]:
class AgriSenseDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(
            Path(row["actual_image_path"])
        ).convert("RGB")

        label = int(row["disease_id"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [9]:
# Create the loaders

train_df = df_clean[
    df_clean["Split"] == "Training"
].copy()

val_df = df_clean[
    df_clean["Split"] == "Validation"
].copy()

test_df = df_clean[
    df_clean["Split"] == "Test"
].copy()

train_dataset = AgriSenseDataset(
    train_df,
    transform=train_transform
)

val_dataset = AgriSenseDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = AgriSenseDataset(
    test_df,
    transform=val_test_transform
)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

5084
811
1504


In [10]:
#Lets calculate the class weights (the only important new part in this experiment)

class_counts = (
    train_df["disease_id"]
    .value_counts()
    .sort_index()
)

class_counts = class_counts.reindex(
    range(len(disease_classes)),
    fill_value=0
)

class_weights = (
    len(train_df)
    / (
        len(disease_classes)
        * class_counts
    )
)

class_weights = torch.tensor(
    class_weights.values,
    dtype=torch.float32
)

print("Number of class weights:", len(class_weights))
print("Minimum weight:", class_weights.min().item())
print("Maximum weight:", class_weights.max().item())

Number of class weights: 115
Minimum weight: 0.19824527204036713
Maximum weight: 22.104347229003906


In [11]:
# Creating the model (the same restnet 18 baseline)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

num_classes = len(disease_classes)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.fc.parameters():
    parameter.requires_grad = True

model = model.to(device)

Device: cpu


In [12]:
# Weighted loss 

class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [13]:
EPOCHS = 5

best_val_loss = float("inf")

checkpoint_dir = (
    project_root
    / "models"
    / "checkpoints"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

latest_checkpoint_path = (
    checkpoint_dir
    / "resnet18_class_weighted_latest.pth"
)

best_checkpoint_path = (
    checkpoint_dir
    / "resnet18_class_weighted_best.pth"
)

history = []

for epoch in range(EPOCHS):

    # Training

    model.train()

    running_loss = 0.0
    train_predictions = []
    train_targets = []

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for images, labels in train_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.detach().cpu().numpy()
        )

    train_loss = (
        running_loss
        / len(train_loader.dataset)
    )

    train_accuracy = accuracy_score(
        train_targets,
        train_predictions
    )

    train_macro_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro",
        zero_division=0
    )

    # Validation
    model.eval()

    val_running_loss = 0.0
    val_predictions = []
    val_targets = []

    val_bar = tqdm(
        val_loader,
        desc="Validation"
    )

    with torch.no_grad():

        for images, labels in val_bar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_running_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )

    val_loss = (
        val_running_loss
        / len(val_loader.dataset)
    )

    val_accuracy = accuracy_score(
        val_targets,
        val_predictions
    )

    val_macro_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro",
        zero_division=0
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "train_macro_f1": train_macro_f1,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Train Macro F1: {train_macro_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

    # Save latest checkpoint
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1
    }, latest_checkpoint_path)

    # Save best checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_macro_f1": val_macro_f1
        }, best_checkpoint_path)

        print("Best checkpoint updated.")

Validation: 100%|██████████| 51/51 [01:05<00:00,  1.28s/it]


Epoch 1/5 | Train Loss: 4.3112 | Train Acc: 0.1221 | Train Macro F1: 0.0900 | Val Loss: 3.5411 | Val Acc: 0.2096 | Val Macro F1: 0.1316
Best checkpoint updated.


Validation: 100%|██████████| 51/51 [01:03<00:00,  1.25s/it]


Epoch 2/5 | Train Loss: 3.0559 | Train Acc: 0.3220 | Train Macro F1: 0.2661 | Val Loss: 2.9898 | Val Acc: 0.3206 | Val Macro F1: 0.2367
Best checkpoint updated.


Validation: 100%|██████████| 51/51 [01:03<00:00,  1.25s/it]


Epoch 3/5 | Train Loss: 2.4661 | Train Acc: 0.4064 | Train Macro F1: 0.3650 | Val Loss: 2.7173 | Val Acc: 0.3674 | Val Macro F1: 0.2943
Best checkpoint updated.


Validation: 100%|██████████| 51/51 [01:04<00:00,  1.26s/it]


Epoch 4/5 | Train Loss: 2.1779 | Train Acc: 0.4467 | Train Macro F1: 0.4098 | Val Loss: 2.7545 | Val Acc: 0.3773 | Val Macro F1: 0.2827


Validation: 100%|██████████| 51/51 [01:04<00:00,  1.26s/it]

Epoch 5/5 | Train Loss: 1.9368 | Train Acc: 0.4782 | Train Macro F1: 0.4536 | Val Loss: 2.6323 | Val Acc: 0.4032 | Val Macro F1: 0.2986
Best checkpoint updated.


In [15]:
#Loading the best weighted model

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])
print("Validation accuracy:", checkpoint["val_accuracy"])
print("Validation Macro F1:", checkpoint["val_macro_f1"])

Best checkpoint epoch: 5
Validation loss: 2.6323217234393965
Validation accuracy: 0.4032059186189889
Validation Macro F1: 0.2986047554004656


In [16]:
#Testing the Weighted models

test_predictions_weighted = []
test_targets_weighted = []

with torch.no_grad():

    test_bar = tqdm(
        test_loader,
        desc="Testing weighted model"
    )

    for images, labels in test_bar:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_predictions_weighted.extend(
            predictions.cpu().numpy()
        )

        test_targets_weighted.extend(
            labels.cpu().numpy()
        )

print(
    "Test samples:",
    len(test_targets_weighted)
)

Testing weighted model: 100%|██████████| 94/94 [01:44<00:00,  1.12s/it]

Test samples: 1504


In [17]:
# Calculating the Three metrices 

weighted_test_accuracy = accuracy_score(
    test_targets_weighted,
    test_predictions_weighted
)

weighted_test_macro_f1 = f1_score(
    test_targets_weighted,
    test_predictions_weighted,
    average="macro",
    zero_division=0
)

weighted_test_weighted_f1 = f1_score(
    test_targets_weighted,
    test_predictions_weighted,
    average="weighted",
    zero_division=0
)

print(f"Weighted Model Test Accuracy: {weighted_test_accuracy:.4f}")
print(f"Weighted Model Test Macro F1: {weighted_test_macro_f1:.4f}")
print(f"Weighted Model Test Weighted F1: {weighted_test_weighted_f1:.4f}")

Weighted Model Test Accuracy: 0.4209
Weighted Model Test Macro F1: 0.3368
Weighted Model Test Weighted F1: 0.4162


In [20]:
#Loading the baseline model again for baseline report

baseline_model = models.resnet18(
    weights=None
)

baseline_model.fc = nn.Linear(
    baseline_model.fc.in_features,
    num_classes
)

baseline_model = baseline_model.to(device)

baseline_checkpoint_path = (
    project_root
    / "models"
    / "checkpoints"
    / "resnet18_baseline_best.pth"
)

baseline_checkpoint = torch.load(
    baseline_checkpoint_path,
    map_location=device
)

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"]
)

baseline_model.eval()

print("Baseline checkpoint epoch:", baseline_checkpoint["epoch"])
print("Baseline validation loss:", baseline_checkpoint["val_loss"])
print("Baseline validation Macro F1:", baseline_checkpoint["val_macro_f1"])

Baseline checkpoint epoch: 5
Baseline validation loss: 2.2170482657840602
Baseline validation Macro F1: 0.29562832708362424


In [21]:
#Generating baseline test prediction again for baseline report

test_predictions_baseline = []
test_targets_baseline = []

with torch.no_grad():

    test_bar = tqdm(
        test_loader,
        desc="Testing baseline model"
    )

    for images, labels in test_bar:

        images = images.to(device)

        outputs = baseline_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_predictions_baseline.extend(
            predictions.cpu().numpy()
        )

        test_targets_baseline.extend(
            labels.numpy()
        )

print(
    "Baseline test samples:",
    len(test_targets_baseline)
)
print(
    "Baseline predictions:",
    len(test_predictions_baseline)
)

Testing baseline model: 100%|██████████| 94/94 [01:43<00:00,  1.10s/it]

Baseline test samples: 1504
Baseline predictions: 1504


In [23]:
# Baseline Report

baseline_report = classification_report(
    test_targets_baseline,
    test_predictions_baseline,
    labels=list(range(num_classes)),
    target_names=[
        id_to_disease[i]
        for i in range(num_classes)
    ],
    output_dict=True,
    zero_division=0
)

baseline_report_df = pd.DataFrame(
    baseline_report
).T.iloc[:num_classes].copy()

baseline_report_df.index.name = "Disease"
baseline_report_df = baseline_report_df.reset_index()

In [24]:
# Weighted model report

weighted_report = classification_report(
    test_targets_weighted,
    test_predictions_weighted,
    labels=list(range(num_classes)),
    target_names=[
        id_to_disease[i]
        for i in range(num_classes)
    ],
    output_dict=True,
    zero_division=0
)

weighted_report_df = pd.DataFrame(
    weighted_report
).T.iloc[:num_classes].copy()

weighted_report_df.index.name = "Disease"
weighted_report_df = weighted_report_df.reset_index()

In [25]:
#comparing both Baseline and Weighted model report

comparison_df = baseline_report_df[
    ["Disease", "f1-score", "support"]
].rename(
    columns={"f1-score": "Baseline F1"}
)

comparison_df["Weighted F1"] = (
    weighted_report_df["f1-score"]
)

comparison_df["F1 Change"] = (
    comparison_df["Weighted F1"]
    - comparison_df["Baseline F1"]
)

comparison_df = comparison_df.sort_values(
    "F1 Change",
    ascending=False
)

display(comparison_df.head(15))

,Disease,Baseline F1,support,Weighted F1,F1 Change
2,apple rust,0.000000,22.0,0.400000,0.400000
55,garlic rust,0.200000,18.0,0.509804,0.309804
12,bean mosaic virus,0.000000,9.0,0.296296,0.296296
35,celery early blight,0.000000,5.0,0.285714,0.285714
19,blueberry botrytis blight,0.000000,3.0,0.285714,0.285714
112,zucchini downy mildew,0.000000,5.0,0.285714,0.285714
37,cherry powdery mildew,0.000000,6.0,0.250000,0.250000
24,broccoli downy mildew,0.000000,3.0,0.250000,0.250000
63,lettuce mosaic virus,0.285714,6.0,0.500000,0.214286
81,rice blast,0.264706,11.0,0.470588,0.205882


In [26]:
display(comparison_df.tail(15))

,Disease,Baseline F1,support,Weighted F1,F1 Change
46,corn rust,0.360000,23.0,0.153846,-0.206154
75,potato early blight,0.214286,9.0,0.000000,-0.214286
105,wheat leaf rust,0.347826,14.0,0.125000,-0.222826
8,banana cordana leaf spot,0.428571,8.0,0.200000,-0.228571
90,strawberry anthracnose,0.666667,7.0,0.416667,-0.250000
29,carrot alternaria leaf blight,0.521739,7.0,0.264151,-0.257588
69,peach scab,0.555556,11.0,0.285714,-0.269841
43,coffee leaf rust,0.651163,21.0,0.370370,-0.280792
89,squash powdery mildew,0.338462,34.0,0.055556,-0.282906
18,blueberry anthracnose,0.666667,4.0,0.375000,-0.291667


In [27]:
#Compare F1 change across test-set support groups

comparison_df["Support Group"] = pd.cut(
    comparison_df["support"],
    bins=[0, 5, 10, 20, 50, float("inf")],
    labels=["1-5", "6-10", "11-20", "21-50", "51+"]
)

support_summary = (
    comparison_df
    .groupby("Support Group", observed=True)
    .agg(
        Diseases=("Disease", "count"),
        Mean_Baseline_F1=("Baseline F1", "mean"),
        Mean_Weighted_F1=("Weighted F1", "mean"),
        Mean_F1_Change=("F1 Change", "mean")
    )
    .reset_index()
)

display(support_summary)

,Support Group,Diseases,Mean_Baseline_F1,Mean_Weighted_F1,Mean_F1_Change
0,1-5,31,0.193604,0.219675,0.026071
1,6-10,27,0.343398,0.312145,-0.031253
2,11-20,35,0.376899,0.368401,-0.008498
3,21-50,20,0.478509,0.476850,-0.001659
4,51+,1,0.713376,0.728682,0.015306


In [28]:
# Count how many diseases improved vs declined

improved = (comparison_df["F1 Change"] > 0).sum()
unchanged = (comparison_df["F1 Change"] == 0).sum()
declined = (comparison_df["F1 Change"] < 0).sum()

print("Diseases improved:", improved)
print("Diseases unchanged:", unchanged)
print("Diseases declined:", declined)

Diseases improved: 52
Diseases unchanged: 16
Diseases declined: 47


## Experiment 1 Conclusion — Class-Weighted CrossEntropy

Class-weighted CrossEntropy was evaluated as an approach to address disease-class imbalance.

The experiment used the same cleaned dataset, train/validation/test splits, preprocessing pipeline, ResNet-18 architecture, frozen backbone, batch size, optimizer, and learning rate as the baseline. The only change was the use of training-set-derived class weights in CrossEntropyLoss.

### Overall Test Performance

| Metric      | Baseline | Class-Weighted | Change  |
|-------------|----------|----------------|---------|
|  Accuracy   |  0.4402  |     0.4209     | -0.0193 |
|  Macro F1   |  0.3399  |     0.3368     | -0.0031 |
| Weighted F1 |  0.4237  |     0.4162     | -0.0075 |

Although 52 of the 115 disease classes improved in F1, 47 declined and 16 remained unchanged.

The rarest test-support group (1–5 samples) showed a positive mean F1 change of +0.0261. However, the 6–10, 11–20, and 21–50 support groups showed negative mean changes.

### Decision

Class weighting improved performance for some rare disease classes but did not improve overall test performance. Therefore, the baseline loss configuration is retained for the current classifier.

This experiment demonstrates that class weighting was evaluated empirically rather than assumed to be beneficial.